# Reading and writing data on Cloud object storage 
Reading from and writing to Cloud object storage (e.g. AWS S3, Google Cloud Storage, Azure Blob Storage) is a bit different than regular filesystems.   Here we access public read buckets and write to an S3-API-compatible Pangeo@EOSC MinIO bucket.  We use `fsspec`, which makes many types of data storage (including S3) look like filesystems. 

In [2]:
import fsspec
import pandas as pd
import os
import xarray as xr

In [3]:
import warnings
warnings.filterwarnings("ignore", category=UserWarning)
from zarr.errors import UnstableSpecificationWarning
warnings.filterwarnings("ignore", category=UnstableSpecificationWarning)

List files on a public read bucket

In [4]:
fs = fsspec.filesystem('s3', anon=True)

In [5]:
fs.ls('anaconda-public-datasets')

['anaconda-public-datasets/data-uber-r',
 'anaconda-public-datasets/enron-email',
 'anaconda-public-datasets/fashion-mnist',
 'anaconda-public-datasets/gdelt',
 'anaconda-public-datasets/iris',
 'anaconda-public-datasets/nyc-taxi',
 'anaconda-public-datasets/reddit',
 'anaconda-public-datasets/tufts']

Reading CSV from a public read bucket

In [6]:
df = pd.read_csv(fs.open("s3://anaconda-public-datasets/iris/iris.csv"))
df

,5.1,3.5,1.4,0.2,Iris-setosa
0,4.9,3.0,1.4,0.2,Iris-setosa
1,4.7,3.2,1.3,0.2,Iris-setosa
2,4.6,3.1,1.5,0.2,Iris-setosa
3,5.0,3.6,1.4,0.2,Iris-setosa
4,5.4,3.9,1.7,0.4,Iris-setosa
...,...,...,...,...,...
144,6.7,3.0,5.2,2.3,Iris-virginica
145,6.3,2.5,5.0,1.9,Iris-virginica
146,6.5,3.0,5.2,2.0,Iris-virginica
147,6.2,3.4,5.4,2.3,Iris-virginica


Write CSV to an S3 bucket

In [ ]:
from dotenv import load_dotenv
_ = load_dotenv(f'{os.environ["HOME"]}/dotenv/protocoast.env')
#_ = load_dotenv(f'{os.environ["HOME"]}/dotenv/test-rps.env')

In [32]:
f'{os.environ["HOME"]}/dotenv/protocoast.env'

'/home/jovyan/dotenv/protocoast.env'

In [14]:
username = os.environ['JUPYTERHUB_USER']
print(username)

jpoehls


In [15]:
fs = fsspec.filesystem('s3', anon=False, skip_instance_cache=True, use_listings_cache=False,
                       endpoint_url=os.environ['ENDPOINT_URL'])

In [27]:
my_bucket = f's3://jerans-big-bucket'

In [18]:
outfile = fs.open(f"{my_bucket}/testing/iris.csv", 
                      mode='wt')

with outfile as f:
    df.to_csv(f)

List files on restricted S3 bucket

In [29]:
fs.ls(f'{my_bucket}/testing/')

['jerans-big-bucket/testing/iris.csv']

In [30]:
print("AWS_ACCESS_KEY_ID", os.getenv("AWS_ACCESS_KEY_ID"))

AWS_ACCESS_KEY_ID izkzQOJ8qnorQUUliN6b


In [ ]:
os.environ.get("AWS_ACCESS_KEY_ID")

os.environ['AWS_ACCESS_KEY_ID'] = 'izkzQOJ8qnorQUUliN6b'
os.environ['AWS_SECRET_ACCESS_KEY'] = 'vNoRgEdS2cEiqjbWYq1iYSl5np80iUwYfz6TWvT6'

'izkzQOJ8qnorQUUliN6b'

In [ ]:
df = pd.read_csv(fs.open(f"s3://{my_bucket}/testing/iris.csv"))
df

The rest of the examples will use xarray, which follows the NetCDF data model

Read NetCDF data from THREDDS OPeNDAP Service  

In [20]:
ds = xr.open_dataset('http://thredds.socib.es/thredds/dodsC/mooring/temperature_recorder/station_andratx-scb_temprec002/L1/dep0001_station-andratx_scb-temprec002_L1_latest.nc',
               engine='pydap')

In [21]:
ds

<xarray.Dataset> Size: 1MB
Dimensions:       (time: 73791)
Coordinates:
  * time          (time) datetime64[ns] 590kB 2026-03-01T00:01:00 ... 2026-04...
Data variables:
    station_name  |S128 128B ...
    WTR_TEM       (time) float64 590kB ...
    QC_WTR_TEM    (time) float32 295kB ...
    LAT           float64 8B ...
    LON           float64 8B ...
    DEPTH         float64 8B ...
Attributes: (12/61)
    title:                               Data from instrument SCB-TEMPREC002 ...
    institution:                         SOCIB (Sistema de ObservaciÃ³n y pre...
    netcdf_version:                      3.0
    Conventions:                         CF-1.6
    abstract:                            Deployment of instrument SCB-TEMPREC...
    summary:                             Deployment of instrument SCB-TEMPREC...
    ...                                  ...
    distribution_statement:              Approved for public release. Distrib...
    license:                             Approved for public release. Distrib...
    citation:                            Balearic Island Coastal and Observin...
    acknowledgement:                     Ministerio de ciencia e innovaciÃ³n ...
    update_interval:                     every 600 seconds
    Unlimited_Dimension:                 time

In [22]:
ds['WTR_TEM'] = ds['WTR_TEM'].where(ds['QC_WTR_TEM'] != 4)

Visualation interlude: plot a time range of data with hvplot

In [23]:
import hvplot.xarray

In [24]:

ds['WTR_TEM'].sel(time=slice('2026-04-01','2026-04-15')).hvplot(grid=True)

:Curve   [time]   (WTR_TEM)

Write Xarray Dataset to NetCDF, then upload to Cloud bucket

In [ ]:
del ds['station_name'].attrs['DODS']

In [ ]:
local_file = 'socib_andratx.nc'
ds.to_netcdf(local_file, mode='w')

In [ ]:
s3_url = f'{my_bucket}/testing/{local_file}'
_ = fs.upload(local_file, s3_url)

Read NetCDF data from s3 bucket

In [ ]:
xr.open_dataset(fs.open(s3_url))

Write Xarray Dataset directly to Cloud bucket in Zarr format

In [ ]:
ds.to_zarr(fs.get_mapper(f'{my_bucket}/testing/socib_andratx.zarr'), mode='w')

In [ ]:
xr.open_dataset(fs.get_mapper(f'{my_bucket}/testing/socib_andratx.zarr'), engine='zarr')